# Task B -- which data processing actually helps

A full factorial over the two data-processing levers, measured on a **real holdout**,
so unlike the last sweep every cell here comes back with a score.

| factor | off | on |
|---|---|---|
| `corpus` | D0: Kannada only, 6,362 comments | D1: + Tamil and Malayalam, ~58,000 |
| `vocab` | V0: MuRIL's tokenizer as shipped | V1: extended with frequent word-forms |
| `aux` | plain 6-way head | + auxiliary head on the act axis |

Four encoders from the first two crossed, plus stock MuRIL as the control that uses
neither, times the objective. Ten cells. The leaders then get more seeds, because one
seed resolves about a point at this corpus size and several cells will land inside that.

About 8.5 hours. The budget guard stops it starting anything it cannot finish.

Sidebar: Accelerator `GPU T4 x2` or `GPU P100`, Internet on. Save Version -> Save & Run All.


In [ ]:
import os, subprocess, sys, pathlib
WORK="/kaggle/working/hastika"
if os.path.isdir(WORK+"/.git"):
    subprocess.run(["git","-C",WORK,"pull","--ff-only"], check=True)
else:
    subprocess.run(["git","clone","-q","-b","task-b","--depth","1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git",WORK], check=True)
os.chdir(WORK); os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True); print("cwd:", os.getcwd())
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf pyarrow '
               '"transformers>=4.45,<6"', shell=True, check=True)
import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))


## 1. Fetch the extra adaptation text

Tamil and Malayalam OffensEval-Dravidian: romanized Dravidian in the same script
convention as your data, carrying no HASTIKA labels. Legal external data, and used
only for masked-language modelling, never for supervision.

`--text-only` skips the label mapping, which is the Kannada config's schema and need
not match the others.


In [ ]:
for lang in ("tamil","malayalam"):
    subprocess.run([sys.executable,"-m", "hastika.task_b.fetch_external","--lang",lang,"--text-only"],
                   check=True)
!ls -la data/external/


## 2. Smoke test

Two minutes, fails loudly. Exercises the auxiliary path, which is the newest code here.


In [ ]:
subprocess.run([sys.executable,"-u","-m", "hastika.task_b.train","--tag","smoke","--folds","0",
                "--epochs","1","--limit","200","--aux-weight","0.3"], check=True)


## 3. The grid


In [ ]:
cmd=[sys.executable,"-u","-m", "experiments.task_b.grid",
     "--budget-hours","10.5","--reserve-min","15",
     "--epochs","6","--aux-weight","0.3",
     "--confirm-top","2","--confirm-seeds","43 44",
     "--final-full-fit",
     "--out","/kaggle/working"]
print(" ".join(cmd), flush=True)
p=subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout: sys.stdout.write(line)
p.wait(); print("grid exit", p.returncode)


## 4. Results


In [ ]:
print(pathlib.Path("/kaggle/working/RESULTS.md").read_text())
print("\nzips:")
for z in sorted(pathlib.Path("/kaggle/working/subs").glob("*.zip")):
    print(" ", z.name, f"{z.stat().st_size/1024:.1f} KB")


## 5. How to read it

The `vs control` column is what you came for. It is each cell minus stock MuRIL on the
same 472 rows, so it isolates what the processing did.

**A gap under about one point is noise** at 3,143 rows. That is why the top two get
re-run with three seeds; trust those rows over the single-seed ones.

Watch for the interaction. If `D1_V1` beats both `D1_V0` and `D0_V1` by more than either
beats the control, the two levers compose and you want both. If it lands between them,
they overlap and you take whichever is cheaper.

The holdout numbers to compare against, measured on this same split: stock MuRIL 0.5948,
MLM-adapted MuRIL 0.5972. The TF-IDF floor is 0.5948 five-fold.

`winner_full.zip` is the leading configuration retrained on all 3,143 rows across five
seeds. That is the one to submit. The other zips are holdout models, useful for
inspecting predictions but trained on 15% less data.
